# ReNexa — Dataset Audit, Cleaning & Preprocessing
**Tagline:** Give Waste Its Next Life.

This notebook performs a complete audit, quality check, class analysis,
and ML-ready dataset preparation for all ReNexa raw datasets.

> **No model training happens here.** Training will be done later on Google Colab.

---

## 1 · Environment Check

In [1]:
import sys
print(f"Python: {sys.version}")

required = ["pandas", "numpy", "matplotlib", "PIL", "cv2", "sklearn", "yaml"]
for lib in required:
    try:
        mod = __import__(lib)
        ver = getattr(mod, "__version__", "OK")
        print(f"  {lib:15s} → {ver}")
    except ImportError:
        print(f"  {lib:15s} → NOT FOUND")

print("\nEnvironment check complete.")

Python: 3.11.5 (tags/v3.11.5:cce6ba9, Aug 24 2023, 14:38:34) [MSC v.1936 64 bit (AMD64)]
  pandas          → 3.0.5
  numpy           → 2.4.4
  matplotlib      → 3.10.8
  PIL             → 12.2.0
  cv2             → 4.13.0
  sklearn         → 1.9.0
  yaml            → 6.0.3

Environment check complete.


In [2]:
import sys
!{sys.executable} -m pip install PyYAML

### Imports

In [3]:
import os
import json
import csv
import hashlib
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")            # non-interactive backend
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import yaml

# Paths
PROJECT_ROOT = Path(r"E:\ReNexa")
RAW_DIR      = PROJECT_ROOT / "data" / "raw"
KNOWLEDGE_DIR= PROJECT_ROOT / "data" / "knowledge"
PROC_DIR     = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR  = PROC_DIR / "reports"
PLOTS_DIR    = REPORTS_DIR / "plots"

for d in [PROC_DIR, REPORTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

print("All imports OK.  Directories ready.")

All imports OK.  Directories ready.


## 2 · Dataset Discovery

In [5]:
def discover_datasets(raw_dir):
    """Walk raw_dir and catalogue each dataset folder."""
    datasets = []
    for folder in sorted(raw_dir.iterdir()):
        if not folder.is_dir():
            continue
        info = {
            "dataset_name": folder.name,
            "folder_path": str(folder),
            "images": [],
            "annotations": [],
            "metadata": [],
            "subfolders": [],
        }
        for root, dirs, files in os.walk(folder):
            root_p = Path(root)
            for f in files:
                fp = root_p / f
                ext = fp.suffix.lower()
                if ext in IMAGE_EXTS:
                    info["images"].append(str(fp))
                elif ext in {".json", ".xml", ".txt", ".csv", ".yaml", ".yml"}:
                    info["annotations"].append(str(fp))
            for d in dirs:
                info["subfolders"].append(str(root_p / d))
        info["image_count"] = len(info["images"])
        info["annotation_count"] = len(info["annotations"])
        datasets.append(info)
    return datasets

raw_datasets = discover_datasets(RAW_DIR)

print(f"Found {len(raw_datasets)} dataset folders in {RAW_DIR}\n")
for ds in raw_datasets:
    print(f"  {ds['dataset_name']:30s}  images={ds['image_count']:>6,}  annotation_files={ds['annotation_count']:>5,}")

Found 6 dataset folders in E:\ReNexa\data\raw

  asian_waste                     images=12,627  annotation_files=12,630
  mju_waste                       images= 7,425  annotation_files=    5
  mmorwad                         images= 3,300  annotation_files=    5
  taco                            images= 1,500  annotation_files=   21
  trashnet                        images= 2,527  annotation_files=    0
  zerowaste-f-final               images= 9,006  annotation_files=    3


## 3 · Dataset Audit Summary

In [6]:
def detect_annotation_format(ds):
    """Heuristic to determine the primary annotation format."""
    ann_files = [Path(a) for a in ds["annotations"]]
    exts = Counter(a.suffix.lower() for a in ann_files)
    names = [a.name.lower() for a in ann_files]

    if any("annotations.json" in n for n in names):
        return "COCO"
    if any("labels.json" in n for n in names):
        return "COCO / Custom JSON"
    if any("data.yaml" in n for n in names):
        if exts.get(".txt", 0) > 5:
            return "YOLO"
    if exts.get(".xml", 0) > 5:
        return "Pascal VOC"
    if exts.get(".txt", 0) > 5:
        return "YOLO"
    if exts.get(".json", 0) > 0:
        return "JSON"
    if exts.get(".csv", 0) > 0:
        return "CSV"
    return "Unknown"


def has_split(ds, split_name):
    """Check if dataset has a split sub-folder."""
    for sf in ds["subfolders"]:
        if split_name in Path(sf).name.lower():
            return True
    return False


audit_rows = []
for ds in raw_datasets:
    fmt = detect_annotation_format(ds)
    row = {
        "dataset": ds["dataset_name"],
        "path": ds["folder_path"],
        "image_count": ds["image_count"],
        "annotation_count": ds["annotation_count"],
        "class_count": "",
        "class_names": "",
        "annotation_format": fmt,
        "train_available": has_split(ds, "train"),
        "validation_available": has_split(ds, "val"),
        "test_available": has_split(ds, "test"),
        "attribute_data_available": "",
        "notes": "",
    }
    audit_rows.append(row)

# WaDaBa note
audit_rows.append({
    "dataset": "wadaba",
    "path": "N/A",
    "image_count": 0,
    "annotation_count": 0,
    "class_count": 0,
    "class_names": "",
    "annotation_format": "N/A",
    "train_available": False,
    "validation_available": False,
    "test_available": False,
    "attribute_data_available": False,
    "notes": "Not available / skipped (access/licensing restrictions)",
})

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(REPORTS_DIR / "dataset_summary.csv", index=False)
print("Saved: dataset_summary.csv\n")
audit_df

Saved: dataset_summary.csv



,dataset,path,image_count,annotation_count,class_count,class_names,annotation_format,train_available,validation_available,test_available,attribute_data_available,notes
0,asian_waste,E:\ReNexa\data\raw\asian_waste,12627,12630,,,YOLO,True,True,True,,
1,mju_waste,E:\ReNexa\data\raw\mju_waste,7425,5,,,Unknown,False,False,False,,
2,mmorwad,E:\ReNexa\data\raw\mmorwad,3300,5,,,COCO,False,False,False,,
3,taco,E:\ReNexa\data\raw\taco,1500,21,,,COCO,False,False,False,,
4,trashnet,E:\ReNexa\data\raw\trashnet,2527,0,,,Unknown,False,False,False,,
5,zerowaste-f-final,E:\ReNexa\data\raw\zerowaste-f-final,9006,3,,,COCO / Custom JSON,True,True,True,,
6,wadaba,N/A,0,0,0,,N/A,False,False,False,False,Not available / skipped (access/licensing rest...


## 4 · Class / Label Analysis

### 4.1 TrashNet classes

In [7]:
trashnet_root = None
for ds in raw_datasets:
    if "trashnet" in ds["dataset_name"].lower():
        trashnet_root = Path(ds["folder_path"])
        break

all_class_rows = []

if trashnet_root:
    # TrashNet: folder-per-class inside dataset-resized
    tn_base = trashnet_root / "dataset-resized"
    if not tn_base.exists():
        # try direct sub-folders
        tn_base = trashnet_root
    for cls_dir in sorted(tn_base.iterdir()):
        if cls_dir.is_dir() and not cls_dir.name.startswith("."):
            count = len([f for f in cls_dir.iterdir() if f.suffix.lower() in IMAGE_EXTS])
            all_class_rows.append({
                "dataset": "trashnet", "original_class": cls_dir.name,
                "count": count, "annotation_type": "folder_name",
                "possible_renexa_class": "", "mapping_status": "PENDING", "notes": ""
            })
            print(f"  trashnet / {cls_dir.name}: {count} images")
else:
    print("TrashNet folder not found.")

  trashnet / cardboard: 403 images
  trashnet / glass: 501 images
  trashnet / metal: 410 images
  trashnet / paper: 594 images
  trashnet / plastic: 482 images
  trashnet / trash: 137 images


### 4.2 TACO classes (COCO format)

In [8]:
taco_root = None
for ds in raw_datasets:
    if "taco" in ds["dataset_name"].lower():
        taco_root = Path(ds["folder_path"])
        break

taco_ann = None
if taco_root:
    ann_path = taco_root / "TACO" / "data" / "annotations.json"
    if ann_path.exists():
        with open(ann_path, "r") as f:
            taco_ann = json.load(f)

if taco_ann:
    taco_cats = {c["id"]: c["name"] for c in taco_ann["categories"]}
    # optionally grab supercategory
    taco_super = {c["id"]: c.get("supercategory", "") for c in taco_ann["categories"]}
    cat_counts = Counter(a["category_id"] for a in taco_ann["annotations"])
    for cid, cnt in cat_counts.most_common():
        all_class_rows.append({
            "dataset": "taco", "original_class": taco_cats.get(cid, str(cid)),
            "count": cnt, "annotation_type": "COCO",
            "possible_renexa_class": "", "mapping_status": "PENDING",
            "notes": f"supercategory={taco_super.get(cid, '')}"
        })
    print(f"TACO: {len(taco_cats)} categories, {len(taco_ann['annotations'])} annotations, {len(taco_ann['images'])} images")
else:
    print("TACO annotations not found.")

TACO: 60 categories, 4784 annotations, 1500 images


### 4.3 MMoRWaD classes (CSV + COCO)

In [12]:
mmorwad_root = None

for ds in raw_datasets:
    if "mmorwad" in ds["dataset_name"].lower():
        mmorwad_root = Path(ds["folder_path"])
        break

if mmorwad_root:
    mmorwad_csv = pd.read_csv(mmorwad_root / "dataset.csv")

    mat_counts = mmorwad_csv["Material"].value_counts()

    for mat, cnt in mat_counts.items():
        all_class_rows.append({
            "dataset": "mmorwad",
            "original_class": mat,
            "count": cnt,
            "annotation_type": "CSV (material)",
            "possible_renexa_class": "",
            "mapping_status": "PENDING",
            "notes": ""
        })

    print(
        f"MMoRWaD: {len(mat_counts)} materials, "
        f"{len(mmorwad_csv)} samples"
    )

    # COCO annotations
    coco_path = mmorwad_root / "coco_annotations.json"

    if coco_path.exists():
        with open(coco_path, "r", encoding="utf-8") as f:
            mmorwad_coco = json.load(f)

        mmorwad_coco_cats = {
            c["id"]: c["name"]
            for c in mmorwad_coco["categories"]
        }

        print(
            f"  COCO categories: "
            f"{list(mmorwad_coco_cats.values())}"
        )
else:
    print("MMoRWaD folder not found.")

MMoRWaD: 18 materials, 3300 samples


KeyError: 'name'

In [13]:
# Inspect the actual MMoRWaD COCO category structure

mmorwad_root = Path(r"E:\ReNexa\data\raw\mmorwad")
coco_path = mmorwad_root / "coco_annotations.json"

with open(coco_path, "r", encoding="utf-8") as f:
    mmorwad_coco = json.load(f)

print("COCO top-level keys:")
print(list(mmorwad_coco.keys()))

print("\nFirst 5 category records:")

if "categories" in mmorwad_coco:
    for category in mmorwad_coco["categories"][:5]:
        print(category)
else:
    print("No categories section found.")

COCO top-level keys:
['images', 'annotations', 'categories']

First 5 category records:
{'id': 1, 'image_id': 1, 'item': 'zagori water bottle 0.5l', 'material': 'Polyethylene Terephthalate (PET)', 'hue': 'Clear', 'top_cap': 'Yes', 'label_attached': 'Yes', 'contaminates': 'No-N/a', 'deformation': 'No-N/a'}
{'id': 2, 'image_id': 2, 'item': 'zagori water bottle 0.5l', 'material': 'Polyethylene Terephthalate (PET)', 'hue': 'Clear', 'top_cap': 'No-N/a', 'label_attached': 'Yes', 'contaminates': 'No-N/a', 'deformation': 'No-N/a'}
{'id': 3, 'image_id': 3, 'item': 'zagori water bottle 0.5l', 'material': 'Polyethylene Terephthalate (PET)', 'hue': 'Clear', 'top_cap': 'No-N/a', 'label_attached': 'No-N/a', 'contaminates': 'No-N/a', 'deformation': 'No-N/a'}
{'id': 4, 'image_id': 4, 'item': 'zagori water bottle 0.5l', 'material': 'Polyethylene Terephthalate (PET)', 'hue': 'Clear', 'top_cap': 'Yes', 'label_attached': 'No-N/a', 'contaminates': 'No-N/a', 'deformation': 'No-N/a'}
{'id': 5, 'image_id': 5,

### 4.4 Asian Waste Detection classes (YOLO)

In [14]:
asian_root = None
for ds in raw_datasets:
    if "asian" in ds["dataset_name"].lower():
        asian_root = Path(ds["folder_path"])
        break

asian_class_names = []
if asian_root:
    yaml_path = asian_root / "data.yaml"
    if yaml_path.exists():
        with open(yaml_path, "r") as f:
            asian_yaml = yaml.safe_load(f)
        asian_class_names = asian_yaml.get("names", [])
        print(f"Asian Waste: {len(asian_class_names)} classes: {asian_class_names}")
    # Count labels
    label_counts = Counter()
    for split in ["train", "valid", "test"]:
        label_dir = asian_root / split / "labels"
        if not label_dir.exists():
            continue
        for lf in label_dir.iterdir():
            if lf.suffix == ".txt":
                try:
                    for line in lf.read_text().strip().split("\n"):
                        if line.strip():
                            cid = int(line.split()[0])
                            label_counts[cid] += 1
                except Exception:
                    pass
    for cid, cnt in label_counts.most_common():
        cname = asian_class_names[cid] if cid < len(asian_class_names) else str(cid)
        all_class_rows.append({
            "dataset": "asian_waste", "original_class": cname,
            "count": cnt, "annotation_type": "YOLO",
            "possible_renexa_class": "", "mapping_status": "PENDING", "notes": ""
        })
else:
    print("Asian Waste folder not found.")

Asian Waste: 9 classes: ['Animal Waste', 'Construction Waste', 'Garbage Bag', 'Glass', 'Metal', 'Organic', 'Paper', 'Plastic', 'waste']


### 4.5 ZeroWaste-f classes

In [15]:
zw_root = None
for ds in raw_datasets:
    if "zerowaste" in ds["dataset_name"].lower():
        zw_root = Path(ds["folder_path"])
        break

zw_class_names = []
if zw_root:
    # Check for labels.json in any split
    for split in ["train", "val", "test"]:
        lj = zw_root / "splits_final_deblurred" / split / "labels.json"
        if lj.exists():
            with open(lj, "r") as f:
                zw_labels = json.load(f)
            # Detectron2-style COCO format
            if "categories" in zw_labels:
                zw_class_names = [c["name"] for c in zw_labels["categories"]]
                cat_map = {c["id"]: c["name"] for c in zw_labels["categories"]}
                ann_counts = Counter(a["category_id"] for a in zw_labels.get("annotations", []))
                for cid, cnt in ann_counts.most_common():
                    all_class_rows.append({
                        "dataset": "zerowaste-f", "original_class": cat_map.get(cid, str(cid)),
                        "count": cnt, "annotation_type": "COCO/Detectron2",
                        "possible_renexa_class": "", "mapping_status": "PENDING", "notes": f"split={split}"
                    })
                print(f"ZeroWaste-f ({split}): {len(zw_labels.get('images',[]))} images, {len(zw_labels.get('annotations',[]))} annotations")
                print(f"  Classes: {zw_class_names}")
            break  # use first found
    if not zw_class_names:
        print("ZeroWaste-f labels.json not found or has no categories.")
else:
    print("ZeroWaste-f folder not found.")

ZeroWaste-f (train): 3002 images, 18002 annotations
  Classes: ['rigid_plastic', 'cardboard', 'metal', 'soft_plastic']


### 4.6 MJU-Waste classes

In [16]:
mju_root = None
for ds in raw_datasets:
    if "mju" in ds["dataset_name"].lower():
        mju_root = Path(ds["folder_path"])
        break

if mju_root:
    # MJU-Waste is single-class segmentation (waste vs background)
    seg_dir = mju_root / "SegmentationClass"
    n_masks = len(list(seg_dir.glob("*.png"))) if seg_dir.exists() else 0
    n_images = len(list((mju_root / "JPEGImages").glob("*.png"))) if (mju_root / "JPEGImages").exists() else 0
    all_class_rows.append({
        "dataset": "mju_waste", "original_class": "waste",
        "count": n_masks, "annotation_type": "segmentation_mask",
        "possible_renexa_class": "", "mapping_status": "PENDING",
        "notes": "single-class segmentation (waste vs background)"
    })
    print(f"MJU-Waste: {n_images} images, {n_masks} masks (single-class: waste)")
else:
    print("MJU-Waste folder not found.")

MJU-Waste: 2475 images, 2475 masks (single-class: waste)


### 4.7 Save all classes CSV

In [17]:
all_classes_df = pd.DataFrame(all_class_rows)
all_classes_df.to_csv(REPORTS_DIR / "all_dataset_classes.csv", index=False)
print(f"Saved: all_dataset_classes.csv  ({len(all_classes_df)} rows)")
all_classes_df.head(20)

Saved: all_dataset_classes.csv  (151 rows)


,dataset,original_class,count,annotation_type,possible_renexa_class,mapping_status,notes
0,trashnet,cardboard,403,folder_name,,PENDING,
1,trashnet,glass,501,folder_name,,PENDING,
2,trashnet,metal,410,folder_name,,PENDING,
3,trashnet,paper,594,folder_name,,PENDING,
4,trashnet,plastic,482,folder_name,,PENDING,
5,trashnet,trash,137,folder_name,,PENDING,
6,taco,Cigarette,667,COCO,,PENDING,supercategory=Cigarette
7,taco,Unlabeled litter,517,COCO,,PENDING,supercategory=Unlabeled litter
8,taco,Plastic film,451,COCO,,PENDING,supercategory=Plastic bag & wrapper
9,taco,Clear plastic bottle,285,COCO,,PENDING,supercategory=Bottle


## 5 · ReNexa Class Mapping

> ⚠️ **REVIEW BEFORE FINAL TRAINING** — Uncertain mappings are marked `REVIEW`.


In [18]:
# Define the proposed ReNexa class mapping
# Format: (dataset, original_class) -> (proposed_renexa_class, decision, reason, confidence)

MAPPING_RULES = {
    # ---- TrashNet ----
    ("trashnet", "cardboard"):  ("cardboard",  "KEEP",  "direct match",         "high"),
    ("trashnet", "glass"):      ("glass",      "KEEP",  "direct match",         "high"),
    ("trashnet", "metal"):      ("metal",      "KEEP",  "direct match",         "high"),
    ("trashnet", "paper"):      ("paper",      "KEEP",  "direct match",         "high"),
    ("trashnet", "plastic"):    ("plastic",    "KEEP",  "direct match",         "high"),
    ("trashnet", "trash"):      ("other",      "REVIEW","ambiguous catch-all",  "low"),

    # ---- Asian Waste ----
    ("asian_waste", "Plastic"):            ("plastic",       "KEEP",   "direct match",              "high"),
    ("asian_waste", "Metal"):              ("metal",         "KEEP",   "direct match",              "high"),
    ("asian_waste", "Glass"):              ("glass",         "KEEP",   "direct match",              "high"),
    ("asian_waste", "Paper"):              ("paper",         "KEEP",   "direct match",              "high"),
    ("asian_waste", "Organic"):            ("organic",       "KEEP",   "direct match",              "high"),
    ("asian_waste", "Garbage Bag"):        ("plastic",       "MERGE",  "garbage bags are plastic",  "medium"),
    ("asian_waste", "Construction Waste"): ("other",         "REVIEW", "mixed material",            "low"),
    ("asian_waste", "Animal Waste"):       ("organic",       "MERGE",  "organic waste",             "medium"),
    ("asian_waste", "waste"):              ("other",         "REVIEW", "ambiguous label",           "low"),

    # ---- ZeroWaste-f ----
    ("zerowaste-f", "rigid_plastic"):  ("plastic",    "MERGE", "rigid plastic → plastic",  "high"),
    ("zerowaste-f", "cardboard"):      ("cardboard",  "KEEP",  "direct match",             "high"),
    ("zerowaste-f", "metal"):          ("metal",      "KEEP",  "direct match",             "high"),
    ("zerowaste-f", "soft_plastic"):   ("plastic",    "MERGE", "soft plastic → plastic",   "medium"),

    # ---- MJU-Waste ----
    ("mju_waste", "waste"): ("other", "REVIEW", "single-class, no material info", "low"),
}

# MMoRWaD materials — group by broad category
MMORWAD_BROAD = {
    "Polyethylene Terephthalate (PET)": ("plastic", "MERGE", "PET → plastic",    "high"),
    "High Density Polyethylene (HDPE)": ("plastic", "MERGE", "HDPE → plastic",   "high"),
    "Polypropylene (PP)":               ("plastic", "MERGE", "PP → plastic",     "high"),
    "Low Density Polyethylene (LDPE)":  ("plastic", "MERGE", "LDPE → plastic",   "high"),
    "Polystyrene (PS)":                 ("plastic", "MERGE", "PS → plastic",     "high"),
    "Polyvinyl Chloride (PVC)":         ("plastic", "MERGE", "PVC → plastic",    "medium"),
    "Other Plastic (Other)":            ("plastic", "MERGE", "other plastic",    "medium"),
    "Aluminium (ALU)":                  ("metal",   "MERGE", "aluminium → metal","high"),
    "Steel (FE)":                       ("metal",   "MERGE", "steel → metal",    "high"),
    "Clear Glass (GL)":                 ("glass",   "MERGE", "clear glass",      "high"),
    "Brown Glass (GL)":                 ("glass",   "MERGE", "brown glass",      "high"),
    "Green Glass (GL)":                 ("glass",   "MERGE", "green glass",      "high"),
    "Paperboard (PAP)":                 ("cardboard","MERGE","paperboard → cardboard","medium"),
    "Cardboard (PAP)":                  ("cardboard","KEEP", "direct match",     "high"),
    "Paper (PAP)":                      ("paper",   "KEEP",  "direct match",     "high"),
    "Paper/Cardboard + Plastic (C/PAP)":           ("other", "REVIEW", "composite",      "low"),
    "Paper/Cardboard + Plastic + Aluminum (C/PAP)":("other", "REVIEW", "composite",      "low"),
    "Plastic + Aluminium (C/PE)":                  ("other", "REVIEW", "composite",      "low"),
}

mapping_rows = []

for _, row in all_classes_df.iterrows():
    ds = row["dataset"]
    cls = row["original_class"]
    key = (ds, cls)

    if key in MAPPING_RULES:
        renexa, dec, reason, conf = MAPPING_RULES[key]
    elif ds == "mmorwad" and cls in MMORWAD_BROAD:
        renexa, dec, reason, conf = MMORWAD_BROAD[cls]
    elif ds == "taco":
        # TACO has many fine-grained classes — mark for review
        renexa, dec, reason, conf = "", "REVIEW", "TACO fine-grained class needs manual mapping", "low"
    else:
        renexa, dec, reason, conf = "", "REVIEW", "no rule defined", "low"

    mapping_rows.append({
        "dataset": ds, "original_class": cls,
        "proposed_renexa_class": renexa, "decision": dec,
        "reason": reason, "confidence_of_mapping": conf, "notes": ""
    })

mapping_df = pd.DataFrame(mapping_rows)
mapping_df.to_csv(REPORTS_DIR / "renexa_class_mapping.csv", index=False)
print(f"Saved: renexa_class_mapping.csv  ({len(mapping_df)} rows)")
mapping_df

Saved: renexa_class_mapping.csv  (151 rows)


,dataset,original_class,proposed_renexa_class,decision,reason,confidence_of_mapping,notes
0,trashnet,cardboard,cardboard,KEEP,direct match,high,
1,trashnet,glass,glass,KEEP,direct match,high,
2,trashnet,metal,metal,KEEP,direct match,high,
3,trashnet,paper,paper,KEEP,direct match,high,
4,trashnet,plastic,plastic,KEEP,direct match,high,
...,...,...,...,...,...,...,...
146,zerowaste-f,cardboard,cardboard,KEEP,direct match,high,
147,zerowaste-f,soft_plastic,plastic,MERGE,soft plastic → plastic,medium,
148,zerowaste-f,rigid_plastic,plastic,MERGE,rigid plastic → plastic,high,
149,zerowaste-f,metal,metal,KEEP,direct match,high,


### ⚠️ REVIEW BEFORE FINAL TRAINING — Mappings needing review

In [19]:
review_df = mapping_df[mapping_df["decision"] == "REVIEW"]
print(f"\n{'='*60}")
print(f"  {len(review_df)} class mappings need REVIEW")
print(f"{'='*60}\n")
review_df[["dataset","original_class","proposed_renexa_class","reason","confidence_of_mapping"]]


  75 class mappings need REVIEW



,dataset,original_class,proposed_renexa_class,reason,confidence_of_mapping
5,trashnet,trash,other,ambiguous catch-all,low
6,taco,Cigarette,,TACO fine-grained class needs manual mapping,low
7,taco,Unlabeled litter,,TACO fine-grained class needs manual mapping,low
8,taco,Plastic film,,TACO fine-grained class needs manual mapping,low
9,taco,Clear plastic bottle,,TACO fine-grained class needs manual mapping,low
...,...,...,...,...,...
128,mmorwad,Paper/Cardboard + Plastic (C/PAP),other,composite,low
133,mmorwad,Plastic + Aluminium (C/PE),other,composite,low
143,asian_waste,waste,other,ambiguous label,low
145,asian_waste,Construction Waste,other,mixed material,low


## 6 · Image Quality Check

Scans **all** images in raw datasets for corruption, zero-byte files, unusual
dimensions, and channel issues. Bad images are recorded but **never deleted**.


In [20]:
def check_image(img_path):
    """Check a single image.  Returns dict with status info."""
    p = Path(img_path)
    result = {
        "file_path": str(p),
        "width": 0, "height": 0, "channels": 0,
        "status": "OK", "reason": ""
    }
    # zero-byte
    try:
        sz = p.stat().st_size
    except Exception:
        result["status"] = "ERROR"
        result["reason"] = "cannot stat file"
        return result
    if sz == 0:
        result["status"] = "ERROR"
        result["reason"] = "zero-byte file"
        return result

    # try opening with Pillow
    try:
        img = Image.open(str(p))
        img.verify()  # check for corruption
        img = Image.open(str(p))  # reopen after verify
        w, h = img.size
        result["width"] = w
        result["height"] = h
        mode = img.mode
        ch_map = {"L": 1, "LA": 2, "RGB": 3, "RGBA": 4, "CMYK": 4, "P": 1}
        result["channels"] = ch_map.get(mode, 0)
        if w < 10 or h < 10:
            result["status"] = "WARNING"
            result["reason"] = "extremely small image"
        if mode == "L":
            result["reason"] = (result["reason"] + " grayscale").strip()
        if mode == "RGBA":
            result["reason"] = (result["reason"] + " has_alpha").strip()
    except Exception as e:
        result["status"] = "ERROR"
        result["reason"] = f"corrupt/unreadable: {str(e)[:80]}"
    return result


# Process all datasets (with progress feedback)
all_img_stats = []
invalid_images = []
SAMPLE_LIMIT = None  # set to e.g. 500 to speed up during testing

for ds in raw_datasets:
    imgs = ds["images"]
    if SAMPLE_LIMIT:
        imgs = imgs[:SAMPLE_LIMIT]
    print(f"Checking {ds['dataset_name']}: {len(imgs)} images ...", end=" ")
    ds_invalid = 0
    for img_path in imgs:
        info = check_image(img_path)
        info["dataset"] = ds["dataset_name"]
        all_img_stats.append(info)
        if info["status"] != "OK":
            invalid_images.append(info)
            ds_invalid += 1
    print(f"  invalid={ds_invalid}")

img_stats_df = pd.DataFrame(all_img_stats)
img_stats_df.to_csv(REPORTS_DIR / "image_statistics.csv", index=False)

invalid_df = pd.DataFrame(invalid_images)
invalid_df.to_csv(REPORTS_DIR / "invalid_images.csv", index=False)

print(f"\nTotal images checked: {len(all_img_stats)}")
print(f"Invalid/warning images: {len(invalid_images)}")
print("Saved: image_statistics.csv, invalid_images.csv")

Checking asian_waste: 12627 images ...   invalid=0
Checking mju_waste: 7425 images ...   invalid=0
Checking mmorwad: 3300 images ...   invalid=0
Checking taco: 1500 images ...   invalid=0
Checking trashnet: 2527 images ...   invalid=0
Checking zerowaste-f-final: 9006 images ...   invalid=0

Total images checked: 36385
Invalid/warning images: 0
Saved: image_statistics.csv, invalid_images.csv


## 8 · Annotation Validation

### 8.1 YOLO annotation check (Asian Waste)

In [21]:
annotation_errors = []

def validate_yolo_label(label_path, dataset_name, n_classes):
    """Validate a single YOLO label file."""
    errors = []
    try:
        text = Path(label_path).read_text().strip()
        if not text:
            return errors  # empty label = no objects (valid)
        for i, line in enumerate(text.split("\n")):
            parts = line.strip().split()
            if len(parts) < 5:
                errors.append({"dataset": dataset_name, "file": str(label_path),
                               "line": i+1, "error": f"too few fields ({len(parts)})"})
                continue
            try:
                cid = int(parts[0])
                xc, yc, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            except ValueError:
                errors.append({"dataset": dataset_name, "file": str(label_path),
                               "line": i+1, "error": "non-numeric values"})
                continue
            if cid < 0 or (n_classes and cid >= n_classes):
                errors.append({"dataset": dataset_name, "file": str(label_path),
                               "line": i+1, "error": f"class_id {cid} out of range [0,{n_classes})"})
            if not (0 <= xc <= 1 and 0 <= yc <= 1):
                errors.append({"dataset": dataset_name, "file": str(label_path),
                               "line": i+1, "error": f"center ({xc},{yc}) outside [0,1]"})
            if w <= 0 or h <= 0:
                errors.append({"dataset": dataset_name, "file": str(label_path),
                               "line": i+1, "error": f"width/height <= 0 ({w},{h})"})
    except Exception as e:
        errors.append({"dataset": dataset_name, "file": str(label_path),
                       "line": 0, "error": str(e)[:100]})
    return errors

if asian_root:
    n_cls = len(asian_class_names)
    for split in ["train", "valid", "test"]:
        label_dir = asian_root / split / "labels"
        if not label_dir.exists():
            continue
        for lf in sorted(label_dir.glob("*.txt")):
            errs = validate_yolo_label(lf, "asian_waste", n_cls)
            annotation_errors.extend(errs)
    print(f"Asian Waste YOLO annotation errors: {len(annotation_errors)}")
else:
    print("Asian Waste not available for validation.")

Asian Waste YOLO annotation errors: 31


### 8.2 COCO annotation check (TACO)

In [22]:
def validate_coco(coco_data, dataset_name):
    """Basic COCO annotation validation."""
    errors = []
    img_ids = {img["id"] for img in coco_data.get("images", [])}
    cat_ids = {c["id"] for c in coco_data.get("categories", [])}

    for ann in coco_data.get("annotations", []):
        if ann.get("image_id") not in img_ids:
            errors.append({"dataset": dataset_name, "file": "annotations.json",
                           "line": ann.get("id", "?"), "error": f"image_id {ann.get('image_id')} not in images"})
        if ann.get("category_id") not in cat_ids:
            errors.append({"dataset": dataset_name, "file": "annotations.json",
                           "line": ann.get("id", "?"), "error": f"category_id {ann.get('category_id')} not in categories"})
        bbox = ann.get("bbox", [])
        if len(bbox) == 4:
            if bbox[2] <= 0 or bbox[3] <= 0:
                errors.append({"dataset": dataset_name, "file": "annotations.json",
                               "line": ann.get("id", "?"), "error": f"bbox width/height <= 0: {bbox}"})
    return errors

if taco_ann:
    taco_errs = validate_coco(taco_ann, "taco")
    annotation_errors.extend(taco_errs)
    print(f"TACO COCO annotation errors: {len(taco_errs)}")

# ZeroWaste-f
if zw_root:
    for split in ["train", "val", "test"]:
        lj = zw_root / "splits_final_deblurred" / split / "labels.json"
        if lj.exists():
            with open(lj, "r") as f:
                zw_data = json.load(f)
            zw_errs = validate_coco(zw_data, f"zerowaste-f ({split})")
            annotation_errors.extend(zw_errs)
            print(f"ZeroWaste-f ({split}) annotation errors: {len(zw_errs)}")

# MMoRWaD COCO
if mmorwad_root:
    coco_path = mmorwad_root / "coco_annotations.json"
    if coco_path.exists():
        with open(coco_path, "r") as f:
            mm_coco = json.load(f)
        mm_errs = validate_coco(mm_coco, "mmorwad")
        annotation_errors.extend(mm_errs)
        print(f"MMoRWaD COCO annotation errors: {len(mm_errs)}")

TACO COCO annotation errors: 0
ZeroWaste-f (train) annotation errors: 0
ZeroWaste-f (val) annotation errors: 0
ZeroWaste-f (test) annotation errors: 0
MMoRWaD COCO annotation errors: 3300


### 8.3 Segmentation mask check (MJU-Waste)

In [23]:
if mju_root:
    img_dir = mju_root / "JPEGImages"
    seg_dir = mju_root / "SegmentationClass"
    if img_dir.exists() and seg_dir.exists():
        img_stems = {p.stem for p in img_dir.glob("*") if p.suffix.lower() in IMAGE_EXTS}
        mask_stems = {p.stem for p in seg_dir.glob("*") if p.suffix.lower() in IMAGE_EXTS}
        missing_masks = img_stems - mask_stems
        extra_masks = mask_stems - img_stems
        if missing_masks:
            for m in list(missing_masks)[:20]:
                annotation_errors.append({"dataset": "mju_waste", "file": m,
                                          "line": 0, "error": "image has no mask"})
        print(f"MJU-Waste: {len(img_stems)} images, {len(mask_stems)} masks")
        print(f"  Missing masks: {len(missing_masks)}, Extra masks: {len(extra_masks)}")

ann_err_df = pd.DataFrame(annotation_errors)
ann_err_df.to_csv(REPORTS_DIR / "annotation_errors.csv", index=False)
print(f"\nTotal annotation errors: {len(annotation_errors)}")
print("Saved: annotation_errors.csv")

MJU-Waste: 2475 images, 2475 masks
  Missing masks: 0, Extra masks: 0

Total annotation errors: 3331
Saved: annotation_errors.csv


## 9 · Dataset-Specific Notes

| Dataset | Type | Format | Notes |
|---------|------|--------|-------|
| TrashNet | Classification | folder-per-class | 6 classes (cardboard, glass, metal, paper, plastic, trash) |
| TACO | Detection/Segmentation | COCO JSON | Real-world, many fine-grained categories |
| MMoRWaD | Classification + Attributes | CSV + COCO JSON | Rich material/condition metadata |
| Asian Waste | Detection | YOLO | 9 classes, train/valid/test splits |
| ZeroWaste-f | Detection/Segmentation | COCO JSON + masks | train/val/test, conveyor-belt images |
| MJU-Waste | Segmentation | VOC masks + split lists | Single-class (waste vs background) |
| WaDaBa | — | — | **Not available / skipped** |


## 10 · Duplicate Image Check

In [24]:
def file_hash(path, chunk_size=8192):
    """Compute MD5 hash of a file."""
    h = hashlib.md5()
    try:
        with open(path, "rb") as f:
            while True:
                chunk = f.read(chunk_size)
                if not chunk:
                    break
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return None

# Build hash index (this can be slow for large datasets)
print("Computing image hashes (this may take a few minutes) ...")
hash_index = defaultdict(list)  # hash -> list of (dataset, path)

for ds in raw_datasets:
    imgs = ds["images"]
    for img_path in imgs:
        h = file_hash(img_path)
        if h:
            hash_index[h].append((ds["dataset_name"], img_path))

# Find duplicates
dup_rows = []
for h, entries in hash_index.items():
    if len(entries) > 1:
        for i, (ds_a, path_a) in enumerate(entries):
            for ds_b, path_b in entries[i+1:]:
                dup_rows.append({
                    "hash": h, "dataset_a": ds_a, "file_a": path_a,
                    "dataset_b": ds_b, "file_b": path_b,
                    "cross_dataset": ds_a != ds_b,
                })

dup_df = pd.DataFrame(dup_rows)
dup_df.to_csv(REPORTS_DIR / "duplicate_images.csv", index=False)
print(f"Duplicate pairs found: {len(dup_df)}")
print(f"  Cross-dataset: {dup_df['cross_dataset'].sum() if len(dup_df) else 0}")
print("Saved: duplicate_images.csv")

Computing image hashes (this may take a few minutes) ...
Duplicate pairs found: 4048
  Cross-dataset: 0
Saved: duplicate_images.csv


## 11 · Data Leakage Check

In [25]:
def get_split_for_file(path_str):
    """Try to determine train/val/test from path."""
    p = path_str.lower().replace("\\", "/")
    if "/train/" in p:
        return "train"
    if "/val/" in p or "/valid/" in p:
        return "val"
    if "/test/" in p:
        return "test"
    return "unknown"

leakage_rows = []
for h, entries in hash_index.items():
    if len(entries) <= 1:
        continue
    splits = [(ds, fp, get_split_for_file(fp)) for ds, fp in entries]
    unique_splits = set(s for _, _, s in splits if s != "unknown")
    if len(unique_splits) > 1:
        for i, (ds_a, fp_a, sp_a) in enumerate(splits):
            for ds_b, fp_b, sp_b in splits[i+1:]:
                if sp_a != sp_b and sp_a != "unknown" and sp_b != "unknown":
                    leakage_rows.append({
                        "dataset": f"{ds_a} / {ds_b}",
                        "file_a": fp_a, "file_b": fp_b,
                        "split_a": sp_a, "split_b": sp_b,
                        "duplicate_type": "exact_hash",
                        "status": "LEAKAGE"
                    })

leakage_df = pd.DataFrame(leakage_rows)
leakage_df.to_csv(REPORTS_DIR / "data_leakage_report.csv", index=False)
print(f"Data leakage instances: {len(leakage_df)}")
print("Saved: data_leakage_report.csv")

Data leakage instances: 1876
Saved: data_leakage_report.csv


## 14 · Attribute Data Preparation (MMoRWaD)

In [27]:
ATTR_DIR = PROC_DIR / "renexa_attributes"
ATTR_DIR.mkdir(parents=True, exist_ok=True)

attr_rows = []
if mmorwad_root:
    mmorwad_csv = pd.read_csv(mmorwad_root / "dataset.csv")
    sample_splits = pd.read_csv(mmorwad_root / "sample_splits.csv")
    # Merge splits
    merged = mmorwad_csv.merge(sample_splits[["ID", "split"]], on="ID", how="left")

    # Parse filename pattern:  IDic_MXXHYYTALDCDDP.png
    # Columns already available: Material, Hue, Top/Cap, Label, Contaminates, Deformation
    for _, row in merged.iterrows():
        attr_rows.append({
            "dataset": "mmorwad",
            "sample_id": row["ID"],
            "item": row["Item"],
            "material": row["Material"],
            "hue": row["Hue"],
            "cap_present": row.get("Top/Cap", ""),
            "label_present": row.get("Label", ""),
            "contamination": row.get("Contaminates", ""),
            "deformation": row.get("Deformation", ""),
            "split": row.get("split", ""),
        })
    print(f"MMoRWaD attributes: {len(attr_rows)} samples")
else:
    print("MMoRWaD not available for attributes.")

attr_df = pd.DataFrame(attr_rows)
if len(attr_df):
    attr_df.to_csv(ATTR_DIR / "mmorwad_attributes.csv", index=False)
    attr_summary = attr_df.describe(include="all").T
    attr_summary.to_csv(REPORTS_DIR / "attribute_summary.csv")
    print("Saved: renexa_attributes/mmorwad_attributes.csv, attribute_summary.csv")
else:
    pd.DataFrame(columns=["note"]).to_csv(REPORTS_DIR / "attribute_summary.csv", index=False)
    print("No attribute data available.")

MMoRWaD attributes: 3300 samples
Saved: renexa_attributes/mmorwad_attributes.csv, attribute_summary.csv


## 15 · Class Imbalance Analysis

In [28]:
balance_rows = []
for ds_name, group in all_classes_df.groupby("dataset"):
    total = group["count"].sum()
    for _, row in group.iterrows():
        pct = round(100 * row["count"] / total, 2) if total > 0 else 0
        balance_rows.append({
            "dataset": ds_name,
            "class": row["original_class"],
            "count": row["count"],
            "percentage": pct,
            "imbalance_ratio": round(group["count"].max() / max(row["count"], 1), 2),
        })

balance_df = pd.DataFrame(balance_rows)
balance_df.to_csv(REPORTS_DIR / "class_balance_report.csv", index=False)
print("Saved: class_balance_report.csv")

# Show most imbalanced
top_imb = balance_df.nlargest(10, "imbalance_ratio")
print("\nTop 10 most imbalanced classes:")
top_imb[["dataset", "class", "count", "percentage", "imbalance_ratio"]]

Saved: class_balance_report.csv

Top 10 most imbalanced classes:


,dataset,class,count,percentage,imbalance_ratio
140,taco,Carded blister pack,1,0.02,667.00
138,taco,Battery,2,0.04,333.50
139,taco,Other plastic cup,2,0.04,333.50
27,mmorwad,Polyvinyl Chloride (PVC),5,0.04,229.20
45,mmorwad,Polyvinyl Chloride (PVC),5,0.04,229.20
63,mmorwad,Polyvinyl Chloride (PVC),5,0.04,229.20
81,mmorwad,Polyvinyl Chloride (PVC),5,0.04,229.20
136,taco,Polypropylene bag,3,0.06,222.33
137,taco,Pizza box,3,0.06,222.33
133,taco,Paper straw,4,0.08,166.75


## 16 · Visualizations

### 16.1 Image count by dataset

In [29]:
fig, ax = plt.subplots(figsize=(10, 5))
ds_names = [ds["dataset_name"] for ds in raw_datasets]
ds_counts = [ds["image_count"] for ds in raw_datasets]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(ds_names)))
ax.barh(ds_names, ds_counts, color=colors)
ax.set_xlabel("Number of Images")
ax.set_title("Image Count by Dataset")
for i, v in enumerate(ds_counts):
    ax.text(v + 10, i, f"{v:,}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "image_count_by_dataset.png", dpi=150)
plt.show()
print("Saved plot: image_count_by_dataset.png")

Saved plot: image_count_by_dataset.png


C:\Users\DELL\AppData\Local\Temp\ipykernel_10008\1680995689.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 16.2 Class distribution per dataset

In [30]:
# Plot top classes per dataset (limit to top 10 per dataset)
unique_ds = all_classes_df["dataset"].unique()
n_ds = len(unique_ds)
fig, axes = plt.subplots(n_ds, 1, figsize=(12, 4 * n_ds))
if n_ds == 1:
    axes = [axes]

for ax, ds_name in zip(axes, unique_ds):
    subset = all_classes_df[all_classes_df["dataset"] == ds_name].nlargest(10, "count")
    ax.barh(subset["original_class"].astype(str), subset["count"],
            color=plt.cm.Set2(np.linspace(0, 1, len(subset))))
    ax.set_title(f"{ds_name} — Top Classes")
    ax.set_xlabel("Count")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "class_distribution_per_dataset.png", dpi=150)
plt.show()
print("Saved plot: class_distribution_per_dataset.png")

Saved plot: class_distribution_per_dataset.png


C:\Users\DELL\AppData\Local\Temp\ipykernel_10008\4108770053.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 16.3 Image dimension distribution

In [31]:
valid_stats = img_stats_df[(img_stats_df["width"] > 0) & (img_stats_df["height"] > 0)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(valid_stats["width"], bins=50, color="steelblue", alpha=0.8)
axes[0].set_title("Width Distribution")
axes[0].set_xlabel("Pixels")

axes[1].hist(valid_stats["height"], bins=50, color="coral", alpha=0.8)
axes[1].set_title("Height Distribution")
axes[1].set_xlabel("Pixels")

aspect = valid_stats["width"] / valid_stats["height"]
axes[2].hist(aspect, bins=50, color="mediumseagreen", alpha=0.8)
axes[2].set_title("Aspect Ratio (W/H)")
axes[2].set_xlabel("Ratio")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "image_dimensions.png", dpi=150)
plt.show()
print("Saved plot: image_dimensions.png")

Saved plot: image_dimensions.png


C:\Users\DELL\AppData\Local\Temp\ipykernel_10008\3381850824.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 16.4 Sample images from each dataset

In [32]:
fig, axes = plt.subplots(len(raw_datasets), 4, figsize=(16, 4 * len(raw_datasets)))
if len(raw_datasets) == 1:
    axes = [axes]

for row_idx, ds in enumerate(raw_datasets):
    imgs = ds["images"][:4]
    for col_idx in range(4):
        ax = axes[row_idx][col_idx]
        if col_idx < len(imgs):
            try:
                img = Image.open(imgs[col_idx]).convert("RGB")
                ax.imshow(img)
                ax.set_title(Path(imgs[col_idx]).name[:25], fontsize=8)
            except Exception:
                ax.text(0.5, 0.5, "Error", ha="center", va="center")
        ax.axis("off")
    axes[row_idx][0].set_ylabel(ds["dataset_name"], fontsize=10, rotation=0,
                                 labelpad=80, va="center")

plt.suptitle("Sample Images from Each Dataset", fontsize=14)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "sample_images.png", dpi=150)
plt.show()
print("Saved plot: sample_images.png")

Saved plot: sample_images.png


C:\Users\DELL\AppData\Local\Temp\ipykernel_10008\3618720524.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 16.5 Sample detection images with bounding boxes (Asian Waste)

In [33]:
def draw_yolo_boxes(img_path, label_path, class_names, ax):
    """Draw YOLO bounding boxes on image."""
    img = cv2.imread(str(img_path))
    if img is None:
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)

    if Path(label_path).exists():
        colors_map = plt.cm.tab10(np.linspace(0, 1, max(len(class_names), 1)))
        for line in Path(label_path).read_text().strip().split("\n"):
            if not line.strip():
                continue
            parts = line.split()
            cid = int(parts[0])
            xc, yc, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            x1 = (xc - bw/2) * w
            y1 = (yc - bh/2) * h
            rect_w = bw * w
            rect_h = bh * h
            color = colors_map[cid % len(colors_map)]
            rect = plt.Rectangle((x1, y1), rect_w, rect_h, linewidth=2,
                                 edgecolor=color, facecolor="none")
            ax.add_patch(rect)
            label = class_names[cid] if cid < len(class_names) else str(cid)
            ax.text(x1, y1-5, label, fontsize=7, color="white",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.7))
    ax.axis("off")

if asian_root:
    train_imgs = sorted((asian_root / "train" / "images").glob("*"))[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for idx, img_p in enumerate(train_imgs):
        ax = axes[idx // 3][idx % 3]
        label_p = asian_root / "train" / "labels" / (img_p.stem + ".txt")
        draw_yolo_boxes(img_p, label_p, asian_class_names, ax)
        ax.set_title(img_p.name[:20], fontsize=8)
    plt.suptitle("Asian Waste — Detection Samples with Bounding Boxes", fontsize=12)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "asian_waste_bbox_samples.png", dpi=150)
    plt.show()
    print("Saved plot: asian_waste_bbox_samples.png")
else:
    print("Asian Waste not available for bbox visualization.")

Saved plot: asian_waste_bbox_samples.png


C:\Users\DELL\AppData\Local\Temp\ipykernel_10008\1264903263.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 16.6 Segmentation samples (MJU-Waste)

In [34]:
if mju_root:
    img_dir = mju_root / "JPEGImages"
    seg_dir = mju_root / "SegmentationClass"
    sample_imgs = sorted(img_dir.glob("*.png"))[:4]
    fig, axes = plt.subplots(len(sample_imgs), 2, figsize=(10, 4 * len(sample_imgs)))
    if len(sample_imgs) == 1:
        axes = [axes]
    for i, img_p in enumerate(sample_imgs):
        mask_p = seg_dir / img_p.name
        try:
            img = Image.open(img_p).convert("RGB")
            axes[i][0].imshow(img)
            axes[i][0].set_title(f"Image: {img_p.name[:25]}", fontsize=8)
        except Exception:
            axes[i][0].text(0.5, 0.5, "Error", ha="center")
        try:
            mask = Image.open(mask_p)
            axes[i][1].imshow(mask, cmap="jet")
            axes[i][1].set_title("Segmentation Mask", fontsize=8)
        except Exception:
            axes[i][1].text(0.5, 0.5, "No mask", ha="center")
        axes[i][0].axis("off")
        axes[i][1].axis("off")
    plt.suptitle("MJU-Waste — Segmentation Samples", fontsize=12)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "mju_waste_seg_samples.png", dpi=150)
    plt.show()
    print("Saved plot: mju_waste_seg_samples.png")
else:
    print("MJU-Waste not available.")

Saved plot: mju_waste_seg_samples.png


C:\Users\DELL\AppData\Local\Temp\ipykernel_10008\529559157.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 16.7 Proposed ReNexa class distribution

In [35]:
# Aggregate counts by proposed ReNexa class (exclude REVIEW/empty)
mapped = mapping_df[mapping_df["proposed_renexa_class"] != ""].copy()
mapped = mapped.merge(all_classes_df[["dataset","original_class","count"]],
                      on=["dataset","original_class"], how="left")
renexa_agg = mapped.groupby("proposed_renexa_class")["count"].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
renexa_agg.plot(kind="barh", ax=ax, color=plt.cm.Paired(np.linspace(0, 1, len(renexa_agg))))
ax.set_xlabel("Total Count (across all datasets)")
ax.set_title("Proposed ReNexa Class Distribution (mapped classes only)")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "renexa_class_distribution.png", dpi=150)
plt.show()
print("Saved plot: renexa_class_distribution.png")

Saved plot: renexa_class_distribution.png


C:\Users\DELL\AppData\Local\Temp\ipykernel_10008\3086640644.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 18 · Train / Validation / Test Split Summary

In [36]:
from sklearn.model_selection import train_test_split

split_rows = []

# TrashNet — no official split, create 80/10/10
if trashnet_root:
    tn_base = trashnet_root / "dataset-resized"
    if not tn_base.exists():
        tn_base = trashnet_root
    all_tn = []
    for cls_dir in sorted(tn_base.iterdir()):
        if cls_dir.is_dir() and not cls_dir.name.startswith("."):
            for f in cls_dir.iterdir():
                if f.suffix.lower() in IMAGE_EXTS:
                    all_tn.append((str(f), cls_dir.name))
    labels = [x[1] for x in all_tn]
    train_data, temp_data, train_lbl, temp_lbl = train_test_split(
        all_tn, labels, test_size=0.2, stratify=labels, random_state=42)
    val_data, test_data, _, _ = train_test_split(
        temp_data, temp_lbl, test_size=0.5, stratify=temp_lbl, random_state=42)
    split_rows.append({"dataset": "trashnet", "train": len(train_data),
                       "val": len(val_data), "test": len(test_data),
                       "split_source": "generated (80/10/10 stratified)"})
    print(f"TrashNet: train={len(train_data)}, val={len(val_data)}, test={len(test_data)}")

# Asian Waste — official splits
if asian_root:
    for split, sname in [("train", "train"), ("valid", "val"), ("test", "test")]:
        d = asian_root / split / "images"
        if d.exists():
            cnt = len(list(d.iterdir()))
            split_rows.append({"dataset": "asian_waste", "split": sname, "count": cnt})
    # consolidate
    aw_splits = {r.get("split", ""): r.get("count", 0)
                 for r in split_rows if r["dataset"] == "asian_waste"}
    split_rows = [r for r in split_rows if r["dataset"] != "asian_waste"]
    split_rows.append({"dataset": "asian_waste",
                       "train": aw_splits.get("train", 0),
                       "val": aw_splits.get("val", 0),
                       "test": aw_splits.get("test", 0),
                       "split_source": "official"})
    print(f"Asian Waste: {aw_splits}")

# ZeroWaste-f — official splits
if zw_root:
    zw_splits = {}
    for split in ["train", "val", "test"]:
        d = zw_root / "splits_final_deblurred" / split / "data"
        if d.exists():
            zw_splits[split] = len(list(d.iterdir()))
    split_rows.append({"dataset": "zerowaste-f",
                       "train": zw_splits.get("train", 0),
                       "val": zw_splits.get("val", 0),
                       "test": zw_splits.get("test", 0),
                       "split_source": "official"})
    print(f"ZeroWaste-f: {zw_splits}")

# MMoRWaD — official splits
if mmorwad_root:
    ss = pd.read_csv(mmorwad_root / "sample_splits.csv")
    mm_splits = ss["split"].value_counts().to_dict()
    split_rows.append({"dataset": "mmorwad",
                       "train": mm_splits.get("train", 0),
                       "val": mm_splits.get("val", 0),
                       "test": mm_splits.get("test", 0),
                       "split_source": "official (brand-disjoint)"})
    print(f"MMoRWaD: {mm_splits}")

# MJU-Waste — official splits from ImageSets
if mju_root:
    mju_splits = {}
    for split in ["train", "val", "test"]:
        sp = mju_root / "ImageSets" / "Segmentation" / f"{split}.txt"
        if sp.exists():
            mju_splits[split] = len(sp.read_text().strip().split("\n"))
    split_rows.append({"dataset": "mju_waste",
                       "train": mju_splits.get("train", 0),
                       "val": mju_splits.get("val", 0),
                       "test": mju_splits.get("test", 0),
                       "split_source": "official"})
    print(f"MJU-Waste: {mju_splits}")

# TACO — no official split
if taco_ann:
    n_taco = len(taco_ann["images"])
    split_rows.append({"dataset": "taco",
                       "train": int(n_taco * 0.8),
                       "val": int(n_taco * 0.1),
                       "test": n_taco - int(n_taco * 0.8) - int(n_taco * 0.1),
                       "split_source": "to be generated (80/10/10)"})
    print(f"TACO: {n_taco} images (split to be generated)")

split_df = pd.DataFrame(split_rows)
split_df.to_csv(REPORTS_DIR / "split_summary.csv", index=False)
print("\nSaved: split_summary.csv")
split_df

TrashNet: train=2021, val=253, test=253
Asian Waste: {'train': 9964, 'val': 1327, 'test': 1336}
ZeroWaste-f: {'train': 3002, 'val': 572, 'test': 929}
MMoRWaD: {'train': 2583, 'test': 360, 'val': 357}
MJU-Waste: {'train': 1485, 'val': 248, 'test': 742}
TACO: 1500 images (split to be generated)

Saved: split_summary.csv


,dataset,train,val,test,split_source
0,trashnet,2021,253,253,generated (80/10/10 stratified)
1,asian_waste,9964,1327,1336,official
2,zerowaste-f,3002,572,929,official
3,mmorwad,2583,357,360,official (brand-disjoint)
4,mju_waste,1485,248,742,official
5,taco,1200,150,150,to be generated (80/10/10)


## 19 · Detection Dataset Preparation

Prepares per-dataset processed outputs.  
**Does not merge** datasets until class mappings are fully resolved.


In [37]:
DET_DIR = PROC_DIR / "renexa_detection"

# Only prepare datasets with resolved (non-REVIEW) mappings
# Asian Waste is the most straightforward YOLO dataset
det_lineage = []

def copy_yolo_dataset(src_root, split_map, class_map, dataset_name, dest_root):
    """Copy a YOLO dataset with class ID remapping.
    split_map: {"train": "train", "valid": "val", "test": "test"}
    class_map: {old_id: new_id} or None to keep original
    """
    copied = 0
    for src_split, dst_split in split_map.items():
        img_src = src_root / src_split / "images"
        lbl_src = src_root / src_split / "labels"
        img_dst = dest_root / dst_split / "images"
        lbl_dst = dest_root / dst_split / "labels"
        img_dst.mkdir(parents=True, exist_ok=True)
        lbl_dst.mkdir(parents=True, exist_ok=True)

        if not img_src.exists():
            continue
        for img_file in img_src.iterdir():
            if img_file.suffix.lower() not in IMAGE_EXTS:
                continue
            # Copy image
            dst_img = img_dst / f"{dataset_name}_{img_file.name}"
            if not dst_img.exists():
                shutil.copy2(str(img_file), str(dst_img))

            # Copy/remap label
            lbl_file = lbl_src / (img_file.stem + ".txt")
            dst_lbl = lbl_dst / f"{dataset_name}_{img_file.stem}.txt"
            if lbl_file.exists() and not dst_lbl.exists():
                lines = lbl_file.read_text().strip().split("\n")
                new_lines = []
                for line in lines:
                    if not line.strip():
                        continue
                    parts = line.split()
                    old_id = int(parts[0])
                    if class_map and old_id in class_map:
                        new_id = class_map[old_id]
                        if new_id is None:
                            continue  # skip excluded classes
                        parts[0] = str(new_id)
                    new_lines.append(" ".join(parts))
                dst_lbl.write_text("\n".join(new_lines))
            copied += 1

            det_lineage.append({
                "source_dataset": dataset_name,
                "original_file": str(img_file),
                "original_class": "",
                "renexa_class": "",
                "processed_file": str(dst_img),
                "status": "copied",
                "notes": f"split={dst_split}"
            })
    return copied

# Check which Asian Waste classes have REVIEW status
aw_review = mapping_df[(mapping_df["dataset"] == "asian_waste") & (mapping_df["decision"] == "REVIEW")]
has_aw_review = len(aw_review) > 0

if asian_root:
    aw_det_dir = DET_DIR / "asian_waste"
    aw_det_dir.mkdir(parents=True, exist_ok=True)

    n = copy_yolo_dataset(
        asian_root,
        {"train": "train", "valid": "val", "test": "test"},
        None,  # keep original class IDs for now
        "aw",
        aw_det_dir,
    )
    print(f"Asian Waste detection: {n} files prepared")
    if has_aw_review:
        print(f"  ⚠️ {len(aw_review)} classes need REVIEW before final merge")
else:
    print("Asian Waste not available.")

print(f"\nDetection data saved to: {DET_DIR}")

Asian Waste detection: 12627 files prepared
  ⚠️ 2 classes need REVIEW before final merge

Detection data saved to: E:\ReNexa\data\processed\renexa_detection


### 20 · YOLO data.yaml

In [38]:
# Create per-dataset data.yaml
# Asian Waste data.yaml — using original classes for now
if asian_root and asian_class_names:
    aw_yaml = {
        "path": str(DET_DIR / "asian_waste"),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "names": {i: name for i, name in enumerate(asian_class_names)},
    }
    yaml_path = DET_DIR / "asian_waste" / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(aw_yaml, f, default_flow_style=False)
    print(f"Created: {yaml_path}")
    print(f"Classes (0-indexed): {asian_class_names}")

    if has_aw_review:
        print("\n⚠️ WARNING: Some classes are marked REVIEW.")
        print("   Final unified data.yaml will be created after review.")
else:
    print("Cannot create data.yaml — Asian Waste not available.")

Created: E:\ReNexa\data\processed\renexa_detection\asian_waste\data.yaml
Classes (0-indexed): ['Animal Waste', 'Construction Waste', 'Garbage Bag', 'Glass', 'Metal', 'Organic', 'Paper', 'Plastic', 'waste']

⚠️ WARNING: Some classes are marked REVIEW.
   Final unified data.yaml will be created after review.


## 21 · Classification Dataset Preparation

In [39]:
CLS_DIR = PROC_DIR / "renexa_classification"

cls_summary_rows = []

# TrashNet — primary source for classification
if trashnet_root:
    tn_base = trashnet_root / "dataset-resized"
    if not tn_base.exists():
        tn_base = trashnet_root

    # Use the splits generated in section 18
    for split_name, split_data in [("train", train_data), ("val", val_data), ("test", test_data)]:
        for img_path, cls_name in split_data:
            dst_dir = CLS_DIR / split_name / cls_name
            dst_dir.mkdir(parents=True, exist_ok=True)
            dst = dst_dir / Path(img_path).name
            if not dst.exists():
                shutil.copy2(img_path, str(dst))

            det_lineage.append({
                "source_dataset": "trashnet",
                "original_file": img_path,
                "original_class": cls_name,
                "renexa_class": cls_name,
                "processed_file": str(dst),
                "status": "copied",
                "notes": f"classification split={split_name}"
            })

    for cls_name in set(x[1] for x in train_data):
        for split_name, split_data in [("train", train_data), ("val", val_data), ("test", test_data)]:
            cnt = sum(1 for _, c in split_data if c == cls_name)
            cls_summary_rows.append({
                "dataset": "trashnet", "class": cls_name,
                "split": split_name, "count": cnt
            })

    print(f"TrashNet classification: train={len(train_data)}, val={len(val_data)}, test={len(test_data)}")
else:
    print("TrashNet not available for classification.")

cls_summary_df = pd.DataFrame(cls_summary_rows)
cls_summary_df.to_csv(REPORTS_DIR / "classification_summary.csv", index=False)
print(f"\nSaved: classification_summary.csv")
print(f"Classification data: {CLS_DIR}")

TrashNet classification: train=2021, val=253, test=253

Saved: classification_summary.csv
Classification data: E:\ReNexa\data\processed\renexa_classification


## 22-24 · Recovery Knowledge Validation

In [40]:
import re

recovery_errors = []

knowledge_files = {
    "recovery_projects.csv": KNOWLEDGE_DIR / "recovery_projects.csv",
    "recovery_projects_master.csv": KNOWLEDGE_DIR / "recovery_projects_master.csv",
    "recovery_projects_needs_verification.csv": KNOWLEDGE_DIR / "recovery_projects_needs_verification.csv",
    "video_index.csv": KNOWLEDGE_DIR / "video_index.csv",
}

YOUTUBE_PATTERN = re.compile(r"https?://(www\.)?youtube\.com/watch\?v=[\w-]+")

for name, path in knowledge_files.items():
    if not path.exists():
        recovery_errors.append({"file": name, "error": "file not found", "row": "", "column": ""})
        print(f"  {name}: NOT FOUND")
        continue

    df = pd.read_csv(path)
    print(f"\n  {name}: {len(df)} rows, {len(df.columns)} columns")
    print(f"    Columns: {list(df.columns)}")

    # Check for duplicates
    if "project_id" in df.columns:
        dups = df[df["project_id"].duplicated(keep=False)]
        if len(dups):
            for pid in dups["project_id"].unique():
                recovery_errors.append({"file": name, "error": "duplicate project_id",
                                        "row": pid, "column": "project_id"})

    # Check important fields
    important_cols = ["waste_item", "material", "product_name", "recovery_type"]
    for col in important_cols:
        if col in df.columns:
            missing = df[col].isna().sum() + (df[col] == "").sum()
            if missing > 0:
                recovery_errors.append({"file": name, "error": f"missing {col}",
                                        "row": f"{missing} rows", "column": col})

    # Check YouTube URLs
    if "youtube_url" in df.columns:
        for idx, row in df.iterrows():
            url = str(row.get("youtube_url", ""))
            if url and url != "nan" and not YOUTUBE_PATTERN.match(url):
                recovery_errors.append({"file": name, "error": "invalid youtube_url format",
                                        "row": idx, "column": "youtube_url"})
        empty_urls = df["youtube_url"].isna().sum() + (df["youtube_url"] == "").sum()
        if empty_urls > 0:
            recovery_errors.append({"file": name, "error": f"missing youtube_url",
                                    "row": f"{empty_urls} rows", "column": "youtube_url"})

    # Check youtube_verified
    if "youtube_verified" in df.columns:
        unverified = (df["youtube_verified"] != "Yes").sum()
        if unverified > 0:
            recovery_errors.append({"file": name, "error": f"unverified videos",
                                    "row": f"{unverified} rows", "column": "youtube_verified"})

recovery_err_df = pd.DataFrame(recovery_errors)
recovery_err_df.to_csv(REPORTS_DIR / "recovery_validation_report.csv", index=False)
print(f"\nRecovery validation issues: {len(recovery_errors)}")
print("Saved: recovery_validation_report.csv")


  recovery_projects.csv: 52 rows, 32 columns
    Columns: ['project_id', 'waste_item', 'material', 'object_type', 'condition', 'product_name', 'product_category', 'recovery_type', 'required_tools', 'additional_materials', 'required_waste_quantity', 'preparation', 'step_1', 'step_2', 'step_3', 'step_4', 'step_5', 'step_6', 'difficulty', 'time_minutes', 'safety_level', 'adult_supervision', 'feasibility', 'skill_level', 'youtube_url', 'youtube_title', 'youtube_channel', 'youtube_verified', 'source_name', 'source_url', 'source_type', 'notes']

  recovery_projects_master.csv: 102 rows, 36 columns
    Columns: ['project_id', 'waste_item', 'material', 'object_type', 'condition', 'product_name', 'product_category', 'recovery_type', 'required_tools', 'additional_materials', 'required_waste_quantity', 'preparation', 'step_1', 'step_2', 'step_3', 'step_4', 'step_5', 'step_6', 'step_7', 'step_8', 'step_9', 'step_10', 'difficulty', 'time_minutes', 'safety_level', 'adult_supervision', 'feasibility'

## 25 · Data Lineage

In [41]:
lineage_df = pd.DataFrame(det_lineage)
lineage_df.to_csv(REPORTS_DIR / "data_lineage.csv", index=False)
print(f"Data lineage entries: {len(lineage_df)}")
print("Saved: data_lineage.csv")

Data lineage entries: 15154
Saved: data_lineage.csv


## 27 · Final Dataset Report

In [44]:
report_lines = []

report_lines.append("=" * 60)
report_lines.append("  ReNexa - Final Dataset Report")
report_lines.append("=" * 60)
report_lines.append("")

# Datasets found
report_lines.append("DATASETS FOUND:")

for ds in raw_datasets:
    report_lines.append(
        f"  [OK] {ds['dataset_name']:30s} images={ds['image_count']:>6,}"
    )

report_lines.append(
    "  [SKIPPED] WaDaBa - Not available (access/licensing)"
)
report_lines.append("")

# Image counts
total_images = sum(ds["image_count"] for ds in raw_datasets)

report_lines.append(f"TOTAL IMAGES: {total_images:,}")
report_lines.append("")

# Annotation formats
report_lines.append("ANNOTATION FORMATS:")

for _, row in audit_df.iterrows():
    if row["dataset"] != "wadaba":
        report_lines.append(
            f"  {row['dataset']:30s} {row['annotation_format']}"
        )

report_lines.append("")

# Quality
report_lines.append(f"INVALID/WARNING IMAGES: {len(invalid_images)}")
report_lines.append(f"ANNOTATION ERRORS: {len(annotation_errors)}")
report_lines.append(f"DUPLICATE PAIRS: {len(dup_df)}")
report_lines.append(f"DATA LEAKAGE INSTANCES: {len(leakage_df)}")
report_lines.append("")

# Class mapping
n_keep = len(mapping_df[mapping_df["decision"] == "KEEP"])
n_merge = len(mapping_df[mapping_df["decision"] == "MERGE"])
n_review = len(mapping_df[mapping_df["decision"] == "REVIEW"])

report_lines.append(
    f"CLASS MAPPINGS: KEEP={n_keep}, MERGE={n_merge}, REVIEW={n_review}"
)

proposed_classes = sorted(
    mapping_df[
        mapping_df["proposed_renexa_class"] != ""
    ]["proposed_renexa_class"].unique()
)

report_lines.append(
    f"PROPOSED RENEXA CLASSES: {proposed_classes}"
)

report_lines.append("")

# Dataset status
report_lines.append(
    "DETECTION DATASET: "
    + ("READY AFTER REVIEW" if has_aw_review else "READY")
)

report_lines.append(
    f"CLASSIFICATION DATASET: "
    f"{'READY' if trashnet_root else 'NOT READY'}"
)

report_lines.append(
    f"ATTRIBUTE DATASET: "
    f"{'READY' if len(attr_rows) else 'NOT READY'}"
)

report_lines.append(
    f"RECOVERY KNOWLEDGE: "
    f"{'READY AFTER REVIEW' if recovery_errors else 'READY'} "
    f"({len(recovery_errors)} issues)"
)

report_lines.append("")
report_lines.append("=" * 60)

report_text = "\n".join(report_lines)

# Explicit UTF-8 encoding fixes the Windows cp1252 error
report_path = REPORTS_DIR / "final_dataset_report.txt"

report_path.write_text(
    report_text,
    encoding="utf-8"
)

print(report_text)
print(f"\nSaved: {report_path}")

  ReNexa - Final Dataset Report

DATASETS FOUND:
  [OK] asian_waste                    images=12,627
  [OK] mju_waste                      images= 7,425
  [OK] mmorwad                        images= 3,300
  [OK] taco                           images= 1,500
  [OK] trashnet                       images= 2,527
  [OK] zerowaste-f-final              images= 9,006
  [SKIPPED] WaDaBa - Not available (access/licensing)

TOTAL IMAGES: 36,385

ANNOTATION FORMATS:
  asian_waste                    YOLO
  mju_waste                      Unknown
  mmorwad                        COCO
  taco                           COCO
  trashnet                       Unknown
  zerowaste-f-final              COCO / Custom JSON

INVALID/WARNING IMAGES: 0
ANNOTATION ERRORS: 3331
DUPLICATE PAIRS: 4048
DATA LEAKAGE INSTANCES: 1876

CLASS MAPPINGS: KEEP=20, MERGE=56, REVIEW=75
PROPOSED RENEXA CLASSES: ['cardboard', 'glass', 'metal', 'organic', 'other', 'paper', 'plastic']

DETECTION DATASET: READY AFTER REVIEW
CLASSIFICA

## 28 · Final ML Readiness Check

In [46]:
print("=" * 60)
print("  ML READINESS CHECK")
print("=" * 60)
print()

checks = {
    "DATASET AUDIT": "PASS" if len(raw_datasets) >= 5 else "REVIEW",
    "IMAGE QUALITY": "PASS" if len(invalid_images) == 0 else "REVIEW",
    "ANNOTATIONS": "PASS" if len(annotation_errors) == 0 else "REVIEW",
    "CLASS MAPPING": "PASS" if n_review == 0 else "REVIEW",
    "DATA LEAKAGE": "PASS" if len(leakage_df) == 0 else "REVIEW",
    "DETECTION DATASET": "READY" if not has_aw_review else "READY AFTER REVIEW",
    "CLASSIFICATION DATASET": "READY" if trashnet_root else "NOT READY",
    "ATTRIBUTE DATASET": "READY" if len(attr_rows) > 0 else "NOT READY",
    "RECOVERY KNOWLEDGE": "READY" if len(recovery_errors) == 0 else "READY AFTER REVIEW",
}

for check, status in checks.items():
    icon = "✓" if "PASS" in status or status == "READY" else "⚠"
    print(f"  {icon} {check:30s}  {status}")

print()
print("=" * 60)
print("  Notebook complete. Ready for Google Colab training stage.")
print("=" * 60)

  ML READINESS CHECK

  ✓ DATASET AUDIT                   PASS
  ✓ IMAGE QUALITY                   PASS
  ⚠ ANNOTATIONS                     REVIEW
  ⚠ CLASS MAPPING                   REVIEW
  ⚠ DATA LEAKAGE                    REVIEW
  ⚠ DETECTION DATASET               READY AFTER REVIEW
  ✓ CLASSIFICATION DATASET          READY
  ✓ ATTRIBUTE DATASET               READY
  ⚠ RECOVERY KNOWLEDGE              READY AFTER REVIEW

  Notebook complete. Ready for Google Colab training stage.


In [47]:
# ReNexa - Review Class Mapping

mapping_path = REPORTS_DIR / "renexa_class_mapping.csv"

mapping_df = pd.read_csv(mapping_path)

print("Total mapping rows:", len(mapping_df))

print("\nColumns:")
print(mapping_df.columns.tolist())

print("\nREVIEW classes:")
review_df = mapping_df[mapping_df["decision"].astype(str).str.upper() == "REVIEW"]

print(review_df.to_string(index=False))

print("\nREVIEW count:", len(review_df))

Total mapping rows: 151

Columns:
['dataset', 'original_class', 'proposed_renexa_class', 'decision', 'reason', 'confidence_of_mapping', 'notes']

REVIEW classes:
    dataset                               original_class proposed_renexa_class decision                                       reason confidence_of_mapping  notes
   trashnet                                        trash                 other   REVIEW                          ambiguous catch-all                   low    NaN
       taco                                    Cigarette                   NaN   REVIEW TACO fine-grained class needs manual mapping                   low    NaN
       taco                             Unlabeled litter                   NaN   REVIEW TACO fine-grained class needs manual mapping                   low    NaN
       taco                                 Plastic film                   NaN   REVIEW TACO fine-grained class needs manual mapping                   low    NaN
       taco                 

In [48]:
# ReNexa - Finalize Class Mapping

final_mapping = {
    # TrashNet
    "trashnet|trash": ("other", "MERGE", "Catch-all waste class mapped to other", "medium"),

    # TACO - Plastic
    "taco|Plastic film": ("plastic", "MERGE", "Plastic material", "high"),
    "taco|Clear plastic bottle": ("plastic", "MERGE", "Plastic material", "high"),
    "taco|Other plastic": ("plastic", "MERGE", "Plastic material", "high"),
    "taco|Other plastic wrapper": ("plastic", "MERGE", "Plastic material", "high"),
    "taco|Plastic bottle cap": ("plastic", "MERGE", "Plastic material", "high"),
    "taco|Plastic straw": ("plastic", "MERGE", "Plastic material", "high"),
    "taco|Disposable plastic cup": ("plastic", "MERGE", "Plastic material", "high"),

    # TACO - Glass
    "taco|Broken glass": ("glass", "MERGE", "Glass material", "high"),
    "taco|Glass bottle": ("glass", "MERGE", "Glass material", "high"),

    # TACO - Metal
    "taco|Drink can": ("metal", "MERGE", "Metal container", "high"),
    "taco|Pop tab": ("metal", "MERGE", "Metal component", "high"),
    "taco|Metal bottle cap": ("metal", "MERGE", "Metal component", "high"),

    # TACO - Paper/Cardboard
    "taco|Other carton": ("cardboard", "MERGE", "Carton/cardboard material", "high"),
    "taco|Normal paper": ("paper", "MERGE", "Paper material", "high"),

    # TACO - Organic
    "taco|Food waste": ("organic", "MERGE", "Organic waste", "high"),

    # TACO - Other
    "taco|Cigarette": ("other", "MERGE", "Non-target household waste item", "high"),
    "taco|Unlabeled litter": ("other", "MERGE", "Unspecified litter", "high"),
    "taco|Styrofoam piece": ("other", "MERGE", "Other household waste material", "high"),

    # Asian Waste
    "asian_waste|Construction Waste": ("other", "MERGE", "Mixed/non-core construction waste", "medium"),

    # MJU-Waste
    "mju_waste|waste": ("other", "MERGE", "Generic waste class without reliable material information", "medium"),
}

# Apply only mappings explicitly defined above
updated_count = 0
unresolved = []

for idx, row in mapping_df.iterrows():

    key = f"{row['dataset']}|{row['original_class']}"

    if key in final_mapping:
        proposed_class, decision, reason, confidence = final_mapping[key]

        mapping_df.at[idx, "proposed_renexa_class"] = proposed_class
        mapping_df.at[idx, "decision"] = decision
        mapping_df.at[idx, "reason"] = reason
        mapping_df.at[idx, "confidence_of_mapping"] = confidence

        updated_count += 1

    elif str(row["decision"]).upper() == "REVIEW":
        unresolved.append(key)

print("Mappings updated:", updated_count)
print("Still unresolved REVIEW classes:", len(unresolved))

if unresolved:
    print("\nUnresolved classes:")
    for item in unresolved:
        print(" -", item)

# Save updated mapping
mapping_path = REPORTS_DIR / "renexa_class_mapping.csv"

mapping_df.to_csv(
    mapping_path,
    index=False,
    encoding="utf-8"
)

print("\nSaved updated mapping:")
print(mapping_path)

# Final counts
print("\nFinal mapping counts:")
print(mapping_df["decision"].value_counts())

print("\nProposed ReNexa classes:")
print(
    sorted(
        mapping_df[
            mapping_df["proposed_renexa_class"].notna()
            & (mapping_df["proposed_renexa_class"] != "")
        ]["proposed_renexa_class"].unique()
    )
)

Mappings updated: 21
Still unresolved REVIEW classes: 54

Unresolved classes:
 - taco|Plastic lid
 - taco|Paper cup
 - taco|Corrugated carton
 - taco|Aluminium foil
 - taco|Single-use carrier bag
 - taco|Other plastic bottle
 - taco|Drink carton
 - taco|Tissues
 - taco|Crisp packet
 - taco|Disposable food container
 - taco|Plastic utensils
 - taco|Food Can
 - taco|Garbage bag
 - taco|Meal carton
 - taco|Rope & strings
 - taco|Paper bag
 - taco|Scrap metal
 - taco|Foam food container
 - taco|Foam cup
 - taco|Magazine paper
 - taco|Wrapping paper
 - taco|Egg carton
 - taco|Aerosol
 - taco|Metal lid
 - taco|Spread tub
 - taco|Squeezable tube
 - taco|Shoe
 - taco|Glass cup
 - taco|Glass jar
 - taco|Aluminium blister pack
 - taco|Other plastic container
 - taco|Toilet tube
 - taco|Six pack rings
 - taco|Paper straw
 - taco|Plastic glooves
 - taco|Tupperware
 - taco|Polypropylene bag
 - taco|Pizza box
 - taco|Battery
 - taco|Other plastic cup
 - taco|Carded blister pack
 - mmorwad|Paper/Card

PermissionError: [Errno 13] Permission denied: 'E:\\ReNexa\\data\\processed\\reports\\renexa_class_mapping.csv'

In [49]:
# Show ALL unresolved REVIEW classes clearly

review_rows = mapping_df[
    mapping_df["decision"].astype(str).str.upper() == "REVIEW"
]

print("Total unresolved REVIEW classes:", len(review_rows))
print()

for _, row in review_rows.iterrows():
    print(f"{row['dataset']} | {row['original_class']}")

Total unresolved REVIEW classes: 54

taco | Plastic lid
taco | Paper cup
taco | Corrugated carton
taco | Aluminium foil
taco | Single-use carrier bag
taco | Other plastic bottle
taco | Drink carton
taco | Tissues
taco | Crisp packet
taco | Disposable food container
taco | Plastic utensils
taco | Food Can
taco | Garbage bag
taco | Meal carton
taco | Rope & strings
taco | Paper bag
taco | Scrap metal
taco | Foam food container
taco | Foam cup
taco | Magazine paper
taco | Wrapping paper
taco | Egg carton
taco | Aerosol
taco | Metal lid
taco | Spread tub
taco | Squeezable tube
taco | Shoe
taco | Glass cup
taco | Glass jar
taco | Aluminium blister pack
taco | Other plastic container
taco | Toilet tube
taco | Six pack rings
taco | Paper straw
taco | Plastic glooves
taco | Tupperware
taco | Polypropylene bag
taco | Pizza box
taco | Battery
taco | Other plastic cup
taco | Carded blister pack
mmorwad | Paper/Cardboard + Plastic + Aluminum (C/PAP)
mmorwad | Paper/Cardboard + Plastic (C/PAP)
mmor

In [50]:
# Save all unresolved REVIEW classes to a text file

review_rows = mapping_df[
    mapping_df["decision"].astype(str).str.upper() == "REVIEW"
]

review_path = REPORTS_DIR / "unresolved_review_classes.txt"

with open(review_path, "w", encoding="utf-8") as f:
    f.write(f"Total unresolved REVIEW classes: {len(review_rows)}\n\n")

    for _, row in review_rows.iterrows():
        f.write(
            f"{row['dataset']} | {row['original_class']}\n"
        )

print(f"Saved: {review_path}")
print(f"Total REVIEW classes: {len(review_rows)}")

Saved: E:\ReNexa\data\processed\reports\unresolved_review_classes.txt
Total REVIEW classes: 54


In [51]:
# ReNexa - Finalize Remaining Class Mapping

# Exact mapping for the remaining REVIEW classes
remaining_mapping = {

    # -------------------------
    # TACO - Plastic
    # -------------------------
    "Plastic lid": "plastic",
    "Single-use carrier bag": "plastic",
    "Other plastic bottle": "plastic",
    "Crisp packet": "plastic",
    "Disposable food container": "plastic",
    "Plastic utensils": "plastic",
    "Garbage bag": "plastic",
    "Spread tub": "plastic",
    "Squeezable tube": "plastic",
    "Other plastic container": "plastic",
    "Six pack rings": "plastic",
    "Plastic glooves": "plastic",
    "Tupperware": "plastic",
    "Polypropylene bag": "plastic",
    "Other plastic cup": "plastic",

    # -------------------------
    # TACO - Paper/Cardboard
    # -------------------------
    "Paper cup": "paper",
    "Corrugated carton": "cardboard",
    "Tissues": "paper",
    "Paper bag": "paper",
    "Magazine paper": "paper",
    "Wrapping paper": "paper",
    "Egg carton": "cardboard",
    "Toilet tube": "cardboard",
    "Paper straw": "paper",
    "Pizza box": "cardboard",

    # -------------------------
    # TACO - Metal
    # -------------------------
    "Aluminium foil": "metal",
    "Food Can": "metal",
    "Scrap metal": "metal",
    "Metal lid": "metal",
    "Aluminium blister pack": "metal",
    "Aerosol": "metal",

    # -------------------------
    # TACO - Glass
    # -------------------------
    "Glass cup": "glass",
    "Glass jar": "glass",

    # -------------------------
    # TACO - Other
    # -------------------------
    "Drink carton": "other",
    "Rope & strings": "other",
    "Foam food container": "other",
    "Foam cup": "other",
    "Shoe": "other",
    "Battery": "other",
    "Carded blister pack": "other",

    # -------------------------
    # MMoRWaD - Composite materials
    # -------------------------
    "Paper/Cardboard + Plastic + Aluminum (C/PAP)": "other",
    "Paper/Cardboard + Plastic (C/PAP)": "other",
    "Plastic + Aluminium (C/PE)": "other",

    # -------------------------
    # Asian Waste
    # -------------------------
    "waste": "other",
}


# Apply mappings to every matching row
updated_count = 0

for idx, row in mapping_df.iterrows():

    dataset = str(row["dataset"]).strip()
    original_class = str(row["original_class"]).strip()

    # Only process REVIEW rows
    if str(row["decision"]).upper() != "REVIEW":
        continue

    # Match TACO classes
    if dataset == "taco" and original_class in remaining_mapping:
        new_class = remaining_mapping[original_class]

    # Match MMoRWaD composite classes
    elif dataset == "mmorwad" and original_class in remaining_mapping:
        new_class = remaining_mapping[original_class]

    # Match Asian Waste generic class
    elif dataset == "asian_waste" and original_class == "waste":
        new_class = "other"

    else:
        continue

    mapping_df.at[idx, "proposed_renexa_class"] = new_class
    mapping_df.at[idx, "decision"] = "MERGE"
    mapping_df.at[idx, "reason"] = (
        "Mapped to the closest ReNexa material category "
        "for unified waste detection"
    )
    mapping_df.at[idx, "confidence_of_mapping"] = "high"

    updated_count += 1


# Check remaining REVIEW rows
remaining_review = mapping_df[
    mapping_df["decision"].astype(str).str.upper() == "REVIEW"
]

print("Mappings updated:", updated_count)
print("Remaining REVIEW classes:", len(remaining_review))

if len(remaining_review) > 0:
    print("\nStill unresolved:")
    for _, row in remaining_review.iterrows():
        print(
            f"{row['dataset']} | {row['original_class']}"
        )


# Save to a NEW file first.
# This avoids the Windows PermissionError from the old open CSV.
final_mapping_path = REPORTS_DIR / "renexa_class_mapping_final.csv"

mapping_df.to_csv(
    final_mapping_path,
    index=False,
    encoding="utf-8"
)

print("\nSaved final mapping:")
print(final_mapping_path)


# Final counts
print("\nFinal mapping counts:")
print(mapping_df["decision"].value_counts())

print("\nFinal ReNexa classes:")

final_classes = sorted(
    mapping_df[
        mapping_df["proposed_renexa_class"].notna()
        & (mapping_df["proposed_renexa_class"] != "")
    ]["proposed_renexa_class"].unique()
)

for i, class_name in enumerate(final_classes):
    print(f"{i}: {class_name}")

Mappings updated: 53
Remaining REVIEW classes: 1

Still unresolved:
taco | Meal carton

Saved final mapping:
E:\ReNexa\data\processed\reports\renexa_class_mapping_final.csv

Final mapping counts:
decision
MERGE     130
KEEP       20
REVIEW      1
Name: count, dtype: int64

Final ReNexa classes:
0: cardboard
1: glass
2: metal
3: organic
4: other
5: paper
6: plastic


In [52]:
# Final unresolved class: TACO Meal carton

mask = (
    (mapping_df["dataset"].astype(str).str.strip() == "taco") &
    (mapping_df["original_class"].astype(str).str.strip() == "Meal carton") &
    (mapping_df["decision"].astype(str).str.upper() == "REVIEW")
)

mapping_df.loc[mask, "proposed_renexa_class"] = "cardboard"
mapping_df.loc[mask, "decision"] = "MERGE"
mapping_df.loc[mask, "reason"] = "Carton-based packaging mapped to cardboard"
mapping_df.loc[mask, "confidence_of_mapping"] = "high"

# Save final mapping
final_mapping_path = REPORTS_DIR / "renexa_class_mapping_final.csv"

mapping_df.to_csv(
    final_mapping_path,
    index=False,
    encoding="utf-8"
)

# Verify
remaining_review = mapping_df[
    mapping_df["decision"].astype(str).str.upper() == "REVIEW"
]

print("Remaining REVIEW classes:", len(remaining_review))

print("\nFinal mapping counts:")
print(mapping_df["decision"].value_counts())

print("\nFinal ReNexa classes:")
for i, class_name in enumerate(
    sorted(mapping_df["proposed_renexa_class"].dropna().unique())
):
    print(f"{i}: {class_name}")

print(f"\nSaved: {final_mapping_path}")

Remaining REVIEW classes: 0

Final mapping counts:
decision
MERGE    131
KEEP      20
Name: count, dtype: int64

Final ReNexa classes:
0: cardboard
1: glass
2: metal
3: organic
4: other
5: paper
6: plastic

Saved: E:\ReNexa\data\processed\reports\renexa_class_mapping_final.csv


In [53]:
# ReNexa - Data Quality Summary

print("=== DATA QUALITY REPORT ===")

print("\nAnnotation errors:")
print(f"Total: {len(annotation_errors)}")

print("\nDuplicate pairs:")
print(f"Total: {len(dup_df)}")

print("\nData leakage instances:")
print(f"Total: {len(leakage_df)}")

print("\nInvalid images:")
print(f"Total: {len(invalid_images)}")

=== DATA QUALITY REPORT ===

Annotation errors:
Total: 3331

Duplicate pairs:
Total: 4048

Data leakage instances:
Total: 1876

Invalid images:
Total: 0


In [54]:
# Show samples from each quality report

print("=== ANNOTATION ERROR SAMPLE ===")
if len(annotation_errors) > 0:
    display(pd.DataFrame(annotation_errors).head(20))

print("\n=== DUPLICATE SAMPLE ===")
if len(dup_df) > 0:
    display(dup_df.head(20))

print("\n=== LEAKAGE SAMPLE ===")
if len(leakage_df) > 0:
    display(leakage_df.head(20))

=== ANNOTATION ERROR SAMPLE ===


,dataset,file,line,error
0,asian_waste,E:\ReNexa\data\raw\asian_waste\train\labels\tr...,1,"width/height <= 0 (0.7057291671875,0.0)"
1,asian_waste,E:\ReNexa\data\raw\asian_waste\train\labels\tr...,1,"width/height <= 0 (0.0,0.23958333281250002)"
2,asian_waste,E:\ReNexa\data\raw\asian_waste\train\labels\tr...,1,"width/height <= 0 (0.0,0.1328125)"
3,asian_waste,E:\ReNexa\data\raw\asian_waste\train\labels\tr...,1,"width/height <= 0 (0.4010416671875,0.0)"
4,asian_waste,E:\ReNexa\data\raw\asian_waste\train\labels\tr...,1,"width/height <= 0 (0.24479166718749998,0.0)"
5,asian_waste,E:\ReNexa\data\raw\asian_waste\train\labels\tr...,1,"width/height <= 0 (0.24739583281250002,0.0)"
6,asian_waste,E:\ReNexa\data\raw\asian_waste\train\labels\tr...,1,"width/height <= 0 (0.8203125,0.0)"
7,asian_waste,E:\ReNexa\data\raw\asian_waste\train\labels\tr...,1,"width/height <= 0 (0.0,0.99453551875)"
8,asian_waste,E:\ReNexa\data\raw\asian_waste\train\labels\tr...,1,"width/height <= 0 (0.0,0.9973958328125001)"
9,asian_waste,E:\ReNexa\data\raw\asian_waste\train\labels\tr...,1,"width/height <= 0 (0.0,0.996296296875)"



=== DUPLICATE SAMPLE ===


,hash,dataset_a,file_a,dataset_b,file_b,cross_dataset
0,4a98bff6034ca405ea4ba2fa334600f8,asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,asian_waste,E:\ReNexa\data\raw\asian_waste\train\images\tr...,False
1,7c1495ad9e50e250d29351ab65cfbffb,asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,asian_waste,E:\ReNexa\data\raw\asian_waste\train\images\tr...,False
2,0298679f481b0dc4a8c2e79481bd9bac,asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,asian_waste,E:\ReNexa\data\raw\asian_waste\train\images\tr...,False
3,12d922f842de12257666541da75abf24,asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,asian_waste,E:\ReNexa\data\raw\asian_waste\train\images\tr...,False
4,2af7aea428240036aa085c98902dd249,asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,asian_waste,E:\ReNexa\data\raw\asian_waste\train\images\tr...,False
5,e78779b7ec59ab7bc28281798d50473b,asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,asian_waste,E:\ReNexa\data\raw\asian_waste\train\images\tr...,False
6,54d410ca5c908b3c654c2e7689db78c8,asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,asian_waste,E:\ReNexa\data\raw\asian_waste\train\images\tr...,False
7,45a74ed477619266ed90cbf68c700e5e,asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,asian_waste,E:\ReNexa\data\raw\asian_waste\train\images\tr...,False
8,c0c9dd19dfb02eb607dc02d88a9b44ab,asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,asian_waste,E:\ReNexa\data\raw\asian_waste\train\images\tr...,False
9,c706b5438728815371031bea5a7a1cba,asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,asian_waste,E:\ReNexa\data\raw\asian_waste\train\images\tr...,False



=== LEAKAGE SAMPLE ===


,dataset,file_a,file_b,split_a,split_b,duplicate_type,status
0,asian_waste / asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,E:\ReNexa\data\raw\asian_waste\train\images\tr...,test,train,exact_hash,LEAKAGE
1,asian_waste / asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,E:\ReNexa\data\raw\asian_waste\train\images\tr...,test,train,exact_hash,LEAKAGE
2,asian_waste / asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,E:\ReNexa\data\raw\asian_waste\train\images\tr...,test,train,exact_hash,LEAKAGE
3,asian_waste / asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,E:\ReNexa\data\raw\asian_waste\train\images\tr...,test,train,exact_hash,LEAKAGE
4,asian_waste / asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,E:\ReNexa\data\raw\asian_waste\train\images\tr...,test,train,exact_hash,LEAKAGE
5,asian_waste / asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,E:\ReNexa\data\raw\asian_waste\train\images\tr...,test,train,exact_hash,LEAKAGE
6,asian_waste / asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,E:\ReNexa\data\raw\asian_waste\train\images\tr...,test,train,exact_hash,LEAKAGE
7,asian_waste / asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,E:\ReNexa\data\raw\asian_waste\train\images\tr...,test,train,exact_hash,LEAKAGE
8,asian_waste / asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,E:\ReNexa\data\raw\asian_waste\train\images\tr...,test,train,exact_hash,LEAKAGE
9,asian_waste / asian_waste,E:\ReNexa\data\raw\asian_waste\test\images\tes...,E:\ReNexa\data\raw\asian_waste\train\images\tr...,test,train,exact_hash,LEAKAGE
